In [1]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("=" * 80)
print("RSNA SCORE IMPROVEMENT — FAST INPUT CHECK")
print("=" * 80)

for folder in sorted(INPUT_ROOT.iterdir()):
    if not folder.is_dir():
        continue

    print(f"\nINPUT: {folder.name}")

    # Only show immediate contents — no recursive scan
    items = list(folder.iterdir())

    for item in items[:30]:
        if item.is_dir():
            print(f"  [DIR]  {item.name}")
        else:
            print(f"  [FILE] {item.name}")

    if len(items) > 30:
        print(f"  ... + {len(items) - 30} more top-level items")

RSNA SCORE IMPROVEMENT — FAST INPUT CHECK

INPUT: competitions
  [DIR]  rsna-knee-abnormality-detection

INPUT: notebooks
  [DIR]  elliotyang37


In [2]:
from pathlib import Path

NOTEBOOK_ROOT = Path("/kaggle/input/notebooks/elliotyang37")

print("=" * 80)
print("PREVIOUS NOTEBOOK OUTPUTS")
print("=" * 80)

for notebook in sorted(NOTEBOOK_ROOT.iterdir()):
    if not notebook.is_dir():
        continue

    print(f"\nNOTEBOOK: {notebook.name}")

    items = list(notebook.iterdir())

    for item in items[:30]:
        if item.is_dir():
            print(f"  [DIR]  {item.name}")
        else:
            print(f"  [FILE] {item.name}")

    if len(items) > 30:
        print(f"  ... + {len(items) - 30} more")

PREVIOUS NOTEBOOK OUTPUTS

NOTEBOOK: rsna-knee-mri-clean-ssl-training
  [FILE] __results__.html
  [DIR]  ssl_training
  [DIR]  ssl_prepared
  [FILE] __notebook__.ipynb
  [DIR]  __results___files
  [FILE] __output__.json
  [FILE] custom.css

NOTEBOOK: rsna-knee-mri-ssl-data-preparation
  [FILE] __results__.html
  [DIR]  rsna_ssl_clean
  [FILE] __notebook__.ipynb
  [DIR]  __results___files
  [FILE] __output__.json
  [FILE] custom.css

NOTEBOOK: rsna-knee-mri-supervised-fine-tuning
  [DIR]  supervised
  [FILE] __results__.html
  [FILE] __notebook__.ipynb
  [FILE] __output__.json
  [FILE] custom.css


In [3]:
from pathlib import Path

ROOT = Path("/kaggle/input/notebooks/elliotyang37")

folders = {
    "DATA PREPARATION":
        ROOT / "rsna-knee-mri-ssl-data-preparation" / "rsna_ssl_clean",

    "SSL PREPARED":
        ROOT / "rsna-knee-mri-clean-ssl-training" / "ssl_prepared",

    "SSL TRAINING":
        ROOT / "rsna-knee-mri-clean-ssl-training" / "ssl_training",

    "SUPERVISED":
        ROOT / "rsna-knee-mri-supervised-fine-tuning" / "supervised",
}

useful_extensions = {
    ".csv", ".parquet", ".pth", ".pt",
    ".ckpt", ".pkl", ".json", ".npy"
}

for name, folder in folders.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    print("Path:", folder)
    print("Exists:", folder.exists())

    if not folder.exists():
        continue

    files = [
        f for f in folder.rglob("*")
        if f.is_file() and f.suffix.lower() in useful_extensions
    ]

    print(f"Useful files found: {len(files)}")

    for f in files[:100]:
        size_mb = f.stat().st_size / (1024 ** 2)

        print(
            f"{f.relative_to(folder)}"
            f"    [{size_mb:.2f} MB]"
        )

    if len(files) > 100:
        print(f"... + {len(files) - 100} more files")


DATA PREPARATION
Path: /kaggle/input/notebooks/elliotyang37/rsna-knee-mri-ssl-data-preparation/rsna_ssl_clean
Exists: True
Useful files found: 299
splits/step05_study_split.csv    [0.29 MB]
splits/step05_series_split.csv    [8.13 MB]
splits/step05_triplets_with_split.parquet    [128.09 MB]
checkpoints/step01_setup.json    [0.00 MB]
checkpoints/step06_preprocessing.json    [0.00 MB]
indexes/step03_ssl_slice_index.parquet    [46.99 MB]
indexes/step02_ssl_series_inventory.csv    [8.00 MB]
indexes/step06_series_percentiles.parquet    [0.92 MB]
indexes/step04_ssl_triplets.parquet    [128.09 MB]
indexes/step06_percentile_shards/percentiles_118.parquet    [0.01 MB]
indexes/step06_percentile_shards/percentiles_050.parquet    [0.01 MB]
indexes/step06_percentile_shards/percentiles_031.parquet    [0.01 MB]
indexes/step06_percentile_shards/percentiles_210.parquet    [0.01 MB]
indexes/step06_percentile_shards/percentiles_236.parquet    [0.01 MB]
indexes/step06_percentile_shards/percentiles_095.par

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

SUP_DIR = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-supervised-fine-tuning/supervised"
)

# ============================================================
# Load baseline artifacts
# ============================================================

studies = pd.read_csv(
    SUP_DIR / "labelled_studies_with_split.csv"
)

metrics = pd.read_csv(
    SUP_DIR / "validation_metrics.csv"
)

preds = pd.read_csv(
    SUP_DIR / "validation_predictions.csv"
)

history = pd.read_csv(
    SUP_DIR / "supervised_training_history.csv"
)

weights = pd.read_csv(
    SUP_DIR / "class_weights.csv"
)

manifest = pd.read_parquet(
    SUP_DIR / "labelled_triplets_with_split.parquet"
)

# ============================================================
# 1. Labelled study split
# ============================================================

print("=" * 80)
print("1. LABELLED STUDY SPLIT")
print("=" * 80)

print("\nShape:", studies.shape)
print("\nColumns:")
print(studies.columns.tolist())

print("\nFirst rows:")
display(studies.head())

# Automatically find split column
split_candidates = [
    c for c in studies.columns
    if "split" in c.lower()
]

print("\nSplit column candidates:", split_candidates)

for c in split_candidates:
    print(f"\n{c}:")
    print(studies[c].value_counts(dropna=False))


# ============================================================
# 2. Triplets per split
# ============================================================

print("\n" + "=" * 80)
print("2. SUPERVISED TRIPLETS")
print("=" * 80)

print("Shape:", manifest.shape)
print("\nColumns:")
print(manifest.columns.tolist())

split_candidates_manifest = [
    c for c in manifest.columns
    if "split" in c.lower()
]

for c in split_candidates_manifest:
    print(f"\n{c}:")
    print(manifest[c].value_counts(dropna=False))


# ============================================================
# 3. Existing validation metrics
# ============================================================

print("\n" + "=" * 80)
print("3. BASELINE VALIDATION METRICS")
print("=" * 80)

display(metrics)


# ============================================================
# 4. Validation predictions
# ============================================================

print("\n" + "=" * 80)
print("4. VALIDATION PREDICTIONS")
print("=" * 80)

print("Shape:", preds.shape)
print("\nColumns:")
print(preds.columns.tolist())

display(preds.head())


# ============================================================
# 5. Training history
# ============================================================

print("\n" + "=" * 80)
print("5. TRAINING HISTORY")
print("=" * 80)

display(history)


# ============================================================
# 6. Existing class weights
# ============================================================

print("\n" + "=" * 80)
print("6. CLASS WEIGHTS")
print("=" * 80)

display(weights)

1. LABELLED STUDY SPLIT

Shape: (58, 15)

Columns:
['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture', 'Split']

First rows:


,StudyInstanceUID,Report,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture,Split
0,1.2.826.0.1.3680043.8.498.10095687747295410396...,Antecedentes Clínicos:\nEsguince rodilla. [DAT...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,train
1,1.2.826.0.1.3680043.8.498.10170898615867673028...,The study reveals normal knee joint alignment...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,train
2,1.2.826.0.1.3680043.8.498.10306159113324811538...,Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,val
3,1.2.826.0.1.3680043.8.498.11287937729196958426...,"MRI of left Knee with \n-3-Plane Loc R'T, Sag ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,train
4,1.2.826.0.1.3680043.8.498.11382021393803389951...,"In the medial compartment, there is longitudi...",1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train



Split column candidates: ['Split']

Split:
Split
train    46
val      12
Name: count, dtype: int64

2. SUPERVISED TRIPLETS
Shape: (9856, 19)

Columns:
['StudyInstanceUID', 'SeriesInstanceUID', 'Anatomical_Plane', 'PreviousPath', 'CentrePath', 'NextPath', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture', 'Split']

Split:
Split
train    7656
val      2200
Name: count, dtype: int64

3. BASELINE VALIDATION METRICS


,Target,AUROC
0,ACL,0.428571
1,MCL,0.700000
2,Medial Meniscus,0.771429
3,Lateral Meniscus,0.593750
4,Medial OA,0.407407
5,Lateral OA,0.800000
6,PF OA,0.375000
7,Effusion,0.771429
8,Synovitis,0.750000
9,Baker's,0.592593



4. VALIDATION PREDICTIONS
Shape: (12, 25)

Columns:
['StudyInstanceUID', 'ACL_true', 'ACL_prob', 'MCL_true', 'MCL_prob', 'Medial Meniscus_true', 'Medial Meniscus_prob', 'Lateral Meniscus_true', 'Lateral Meniscus_prob', 'Medial OA_true', 'Medial OA_prob', 'Lateral OA_true', 'Lateral OA_prob', 'PF OA_true', 'PF OA_prob', 'Effusion_true', 'Effusion_prob', 'Synovitis_true', 'Synovitis_prob', "Baker's_true", "Baker's_prob", 'Contusion_true', 'Contusion_prob', 'Fracture_true', 'Fracture_prob']


,StudyInstanceUID,ACL_true,ACL_prob,MCL_true,MCL_prob,Medial Meniscus_true,Medial Meniscus_prob,Lateral Meniscus_true,Lateral Meniscus_prob,Medial OA_true,...,Effusion_true,Effusion_prob,Synovitis_true,Synovitis_prob,Baker's_true,Baker's_prob,Contusion_true,Contusion_prob,Fracture_true,Fracture_prob
0,1.2.826.0.1.3680043.8.498.10306159113324811538...,0.0,0.625000,0.0,0.450928,1.0,0.654785,0.0,0.643555,1.0,...,0.0,0.799805,1.0,0.603027,0.0,0.449707,0.0,0.441406,0.0,0.599609
1,1.2.826.0.1.3680043.8.498.16060119389060497136...,0.0,0.518066,0.0,0.437256,1.0,0.540527,1.0,0.583008,1.0,...,1.0,0.666016,1.0,0.537598,1.0,0.506836,0.0,0.470215,1.0,0.515625
2,1.2.826.0.1.3680043.8.498.27437263843446879932...,0.0,0.405273,0.0,0.520996,1.0,0.273438,0.0,0.337646,0.0,...,0.0,0.310547,0.0,0.380615,0.0,0.357910,0.0,0.460449,0.0,0.485352
3,1.2.826.0.1.3680043.8.498.47921753480592595198...,0.0,0.673828,0.0,0.543945,1.0,0.680176,0.0,0.559082,0.0,...,1.0,0.770020,1.0,0.593262,0.0,0.418945,0.0,0.583008,1.0,0.722168
4,1.2.826.0.1.3680043.8.498.48946580946665031852...,1.0,0.374268,0.0,0.151611,0.0,0.322266,0.0,0.447998,0.0,...,1.0,0.613770,0.0,0.519043,0.0,0.461182,1.0,0.229736,0.0,0.427490



5. TRAINING HISTORY


,epoch,train_loss,val_loss,encoder_lr,classifier_lr
0,1,0.928826,0.900458,9.755283e-06,0.000098
1,2,0.905813,0.899060,9.045085e-06,0.000090
2,3,0.880654,0.897515,7.938926e-06,0.000079
3,4,0.869071,0.892123,6.545085e-06,0.000065
4,5,0.850768,0.894193,5.000000e-06,0.000050
5,6,0.838805,0.890474,3.454915e-06,0.000035
6,7,0.833181,0.903629,2.061074e-06,0.000021
7,8,0.824891,0.901296,9.549150e-07,0.000010
8,9,0.819882,0.907513,2.447174e-07,0.000002



6. CLASS WEIGHTS


,Target,Positive,Negative,PosWeight
0,ACL,19,27,1.421
1,MCL,7,39,5.571
2,Medial Meniscus,21,25,1.190
3,Lateral Meniscus,19,27,1.421
4,Medial OA,12,34,2.833
5,Lateral OA,9,37,4.111
6,PF OA,17,29,1.706
7,Effusion,28,18,0.643
8,Synovitis,21,25,1.190
9,Baker's,9,37,4.111


In [5]:
TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

print("=" * 85)
print("LABEL DISTRIBUTION AUDIT")
print("=" * 85)

rows = []

for target in TARGETS:

    total_pos = int(studies[target].sum())
    total_neg = len(studies) - total_pos

    train_df = studies[studies["Split"] == "train"]
    val_df = studies[studies["Split"] == "val"]

    train_pos = int(train_df[target].sum())
    train_neg = len(train_df) - train_pos

    val_pos = int(val_df[target].sum())
    val_neg = len(val_df) - val_pos

    rows.append({
        "Target": target,
        "Total_Pos": total_pos,
        "Total_Neg": total_neg,
        "Train_Pos": train_pos,
        "Train_Neg": train_neg,
        "Val_Pos": val_pos,
        "Val_Neg": val_neg,
        "Val_Pos_Rate": val_pos / len(val_df)
    })

label_audit = pd.DataFrame(rows)

display(label_audit)

print("\n" + "=" * 85)
print("FULL DATASET POSITIVE RATES")
print("=" * 85)

for target in TARGETS:
    pos = int(studies[target].sum())
    print(
        f"{target:<18} "
        f"{pos:>2}/58 positive "
        f"({pos / 58:.1%})"
    )

print("\n" + "=" * 85)
print("CURRENT VALIDATION POSITIVE COUNTS")
print("=" * 85)

for target in TARGETS:

    n_pos = int(
        studies.loc[
            studies["Split"] == "val",
            target
        ].sum()
    )

    n_neg = 12 - n_pos

    warning = ""

    if n_pos <= 1:
        warning = "  <-- VERY UNSTABLE"
    elif n_pos <= 2:
        warning = "  <-- unstable"

    print(
        f"{target:<18} "
        f"pos={n_pos:>2}  "
        f"neg={n_neg:>2}"
        f"{warning}"
    )

LABEL DISTRIBUTION AUDIT


,Target,Total_Pos,Total_Neg,Train_Pos,Train_Neg,Val_Pos,Val_Neg,Val_Pos_Rate
0,ACL,24,34,19,27,5,7,0.416667
1,MCL,9,49,7,39,2,10,0.166667
2,Medial Meniscus,26,32,21,25,5,7,0.416667
3,Lateral Meniscus,23,35,19,27,4,8,0.333333
4,Medial OA,15,43,12,34,3,9,0.250000
5,Lateral OA,11,47,9,37,2,10,0.166667
6,PF OA,21,37,17,29,4,8,0.333333
7,Effusion,35,23,28,18,7,5,0.583333
8,Synovitis,27,31,21,25,6,6,0.500000
9,Baker's,12,46,9,37,3,9,0.250000



FULL DATASET POSITIVE RATES
ACL                24/58 positive (41.4%)
MCL                 9/58 positive (15.5%)
Medial Meniscus    26/58 positive (44.8%)
Lateral Meniscus   23/58 positive (39.7%)
Medial OA          15/58 positive (25.9%)
Lateral OA         11/58 positive (19.0%)
PF OA              21/58 positive (36.2%)
Effusion           35/58 positive (60.3%)
Synovitis          27/58 positive (46.6%)
Baker's            12/58 positive (20.7%)
Contusion          19/58 positive (32.8%)
Fracture           18/58 positive (31.0%)

CURRENT VALIDATION POSITIVE COUNTS
ACL                pos= 5  neg= 7
MCL                pos= 2  neg=10  <-- unstable
Medial Meniscus    pos= 5  neg= 7
Lateral Meniscus   pos= 4  neg= 8
Medial OA          pos= 3  neg= 9
Lateral OA         pos= 2  neg=10  <-- unstable
PF OA              pos= 4  neg= 8
Effusion           pos= 7  neg= 5
Synovitis          pos= 6  neg= 6
Baker's            pos= 3  neg= 9
Contusion          pos= 4  neg= 8
Fracture           pos= 4  ne

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# Configuration
# ============================================================

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

N_FOLDS = 5
RANDOM_SEED = 42
N_TRIALS = 50000

rng = np.random.default_rng(RANDOM_SEED)

cv_df = studies.copy().reset_index(drop=True)

Y = cv_df[TARGETS].astype(int).to_numpy()

n = len(cv_df)

# Fold sizes: 58 -> 12,12,12,11,11
base = n // N_FOLDS
remainder = n % N_FOLDS

fold_sizes = np.array([
    base + (1 if f < remainder else 0)
    for f in range(N_FOLDS)
])

print("Fold sizes:", fold_sizes)
print("Total:", fold_sizes.sum())

# ============================================================
# Expected positive counts in each fold
# ============================================================

prevalence = Y.mean(axis=0)

expected_pos = np.outer(
    fold_sizes,
    prevalence
)

# ============================================================
# Search for a balanced multilabel assignment
# ============================================================

best_score = np.inf
best_assignment = None
best_counts = None

for trial in range(N_TRIALS):

    permutation = rng.permutation(n)

    assignment = np.empty(n, dtype=int)

    start = 0

    for fold, size in enumerate(fold_sizes):

        idx = permutation[start:start + size]

        assignment[idx] = fold

        start += size

    # Positive counts for each target within each fold
    counts = np.vstack([
        Y[assignment == fold].sum(axis=0)
        for fold in range(N_FOLDS)
    ])

    # --------------------------------------------------------
    # Hard requirement:
    # every fold needs positive AND negative examples
    # for every target
    # --------------------------------------------------------

    valid = True

    for fold in range(N_FOLDS):

        if np.any(counts[fold] == 0):
            valid = False
            break

        if np.any(counts[fold] == fold_sizes[fold]):
            valid = False
            break

    if not valid:
        continue

    # --------------------------------------------------------
    # Balance objective
    #
    # Rare labels get relatively more importance.
    # --------------------------------------------------------

    scale = np.sqrt(expected_pos + 0.5)

    score = np.mean(
        ((counts - expected_pos) / scale) ** 2
    )

    if score < best_score:

        best_score = score
        best_assignment = assignment.copy()
        best_counts = counts.copy()

# ============================================================
# Check that a valid split was found
# ============================================================

if best_assignment is None:
    raise RuntimeError(
        "No valid 5-fold split found. "
        "Increase N_TRIALS or use a different strategy."
    )

cv_df["Fold"] = best_assignment

print("\n" + "=" * 90)
print("5-FOLD MULTILABEL SPLIT CREATED")
print("=" * 90)

print(f"\nBest balance score: {best_score:.6f}")

print("\nStudies per fold:")
print(
    cv_df["Fold"]
    .value_counts()
    .sort_index()
)

# ============================================================
# Label counts by fold
# ============================================================

rows = []

for fold in range(N_FOLDS):

    fold_data = cv_df[
        cv_df["Fold"] == fold
    ]

    for target in TARGETS:

        positive = int(
            fold_data[target].sum()
        )

        negative = len(fold_data) - positive

        rows.append({
            "Fold": fold,
            "Target": target,
            "Studies": len(fold_data),
            "Positive": positive,
            "Negative": negative,
            "PositiveRate": positive / len(fold_data),
        })

fold_audit = pd.DataFrame(rows)

print("\n" + "=" * 90)
print("POSITIVE COUNTS BY FOLD")
print("=" * 90)

positive_table = (
    fold_audit
    .pivot(
        index="Target",
        columns="Fold",
        values="Positive"
    )
)

positive_table["Total"] = positive_table.sum(axis=1)

display(positive_table)

# ============================================================
# Safety checks
# ============================================================

print("\n" + "=" * 90)
print("CV SAFETY CHECK")
print("=" * 90)

all_valid = True

for fold in range(N_FOLDS):

    fold_data = cv_df[
        cv_df["Fold"] == fold
    ]

    for target in TARGETS:

        pos = int(fold_data[target].sum())
        neg = len(fold_data) - pos

        if pos == 0 or neg == 0:

            print(
                f"WARNING: Fold {fold}, "
                f"{target}: pos={pos}, neg={neg}"
            )

            all_valid = False

if all_valid:
    print(
        "✓ Every target has both positive and "
        "negative cases in every fold."
    )

# ============================================================
# Save
# ============================================================

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "rsna_score_improvement/splits"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

save_path = (
    OUTPUT_DIR /
    "labelled_studies_5fold.csv"
)

cv_df.to_csv(
    save_path,
    index=False
)

print("\nSaved:")
print(save_path)

Fold sizes: [12 12 12 11 11]
Total: 58

5-FOLD MULTILABEL SPLIT CREATED

Best balance score: 0.162796

Studies per fold:
Fold
0    12
1    12
2    12
3    11
4    11
Name: count, dtype: int64

POSITIVE COUNTS BY FOLD


Fold,0,1,2,3,4,Total
Target,,,,,,
ACL,6,5,6,4,3,24
Baker's,2,3,2,2,3,12
Contusion,4,3,4,5,3,19
Effusion,8,7,8,6,6,35
Fracture,4,4,3,3,4,18
Lateral Meniscus,4,4,4,5,6,23
Lateral OA,3,2,1,3,2,11
MCL,3,2,1,2,1,9
Medial Meniscus,5,7,4,6,4,26



CV SAFETY CHECK
✓ Every target has both positive and negative cases in every fold.

Saved:
/kaggle/working/rsna_score_improvement/splits/labelled_studies_5fold.csv


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np

SUP_DIR = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-supervised-fine-tuning/supervised"
)

OUTPUT_DIR = Path(
    "/kaggle/working/rsna_score_improvement/splits"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Load previous labelled triplets
# ============================================================

triplets = pd.read_parquet(
    SUP_DIR / "labelled_triplets_with_split.parquet"
)

print("=" * 85)
print("ORIGINAL LABELLED TRIPLET MANIFEST")
print("=" * 85)

print("Shape:", triplets.shape)
print("Unique studies:", triplets["StudyInstanceUID"].nunique())

print("\nOld split:")
print(triplets["Split"].value_counts())

# Preserve old split only for reference
triplets = triplets.rename(
    columns={"Split": "BaselineSplit"}
)

# ============================================================
# Merge new 5-fold assignment
# ============================================================

fold_map = cv_df[
    ["StudyInstanceUID", "Fold"]
].copy()

cv_manifest = triplets.merge(
    fold_map,
    on="StudyInstanceUID",
    how="left",
    validate="many_to_one"
)

# ============================================================
# Basic checks
# ============================================================

print("\n" + "=" * 85)
print("5-FOLD TRIPLET MANIFEST")
print("=" * 85)

print("Shape:", cv_manifest.shape)

missing_fold = cv_manifest["Fold"].isna().sum()

print("Missing fold assignments:", missing_fold)
print(
    "Unique studies:",
    cv_manifest["StudyInstanceUID"].nunique()
)

assert missing_fold == 0
assert cv_manifest["StudyInstanceUID"].nunique() == 58

# Every study must belong to exactly one fold
study_fold_counts = (
    cv_manifest
    .groupby("StudyInstanceUID")["Fold"]
    .nunique()
)

assert study_fold_counts.max() == 1

print("✓ Every study belongs to exactly one fold.")

# ============================================================
# Fold-level audit
# ============================================================

audit_rows = []

for fold in range(5):

    fold_df = cv_manifest[
        cv_manifest["Fold"] == fold
    ]

    audit_rows.append({
        "Fold": fold,
        "Studies": fold_df["StudyInstanceUID"].nunique(),
        "Series": fold_df["SeriesInstanceUID"].nunique(),
        "Triplets": len(fold_df)
    })

fold_manifest_audit = pd.DataFrame(audit_rows)

print("\nFold manifest summary:")
display(fold_manifest_audit)

# ============================================================
# MRI plane distribution
# ============================================================

print("\n" + "=" * 85)
print("TRIPLETS BY PLANE AND FOLD")
print("=" * 85)

plane_table = pd.crosstab(
    cv_manifest["Anatomical_Plane"],
    cv_manifest["Fold"]
)

plane_table["Total"] = plane_table.sum(axis=1)

display(plane_table)

# ============================================================
# Leakage safety test
# ============================================================

print("\n" + "=" * 85)
print("LEAKAGE CHECK")
print("=" * 85)

for fold in range(5):

    val_studies = set(
        cv_manifest.loc[
            cv_manifest["Fold"] == fold,
            "StudyInstanceUID"
        ]
    )

    train_studies = set(
        cv_manifest.loc[
            cv_manifest["Fold"] != fold,
            "StudyInstanceUID"
        ]
    )

    overlap = val_studies & train_studies

    print(
        f"Fold {fold}: "
        f"train studies={len(train_studies)}, "
        f"val studies={len(val_studies)}, "
        f"overlap={len(overlap)}"
    )

    assert len(overlap) == 0

print("\n✓ NO STUDY-LEVEL LEAKAGE ACROSS ANY FOLD")

# ============================================================
# Save
# ============================================================

manifest_path = (
    OUTPUT_DIR /
    "labelled_triplets_5fold.parquet"
)

audit_path = (
    OUTPUT_DIR /
    "fold_manifest_audit.csv"
)

cv_manifest.to_parquet(
    manifest_path,
    index=False
)

fold_manifest_audit.to_csv(
    audit_path,
    index=False
)

print("\nSaved:")
print(manifest_path)
print(audit_path)

ORIGINAL LABELLED TRIPLET MANIFEST
Shape: (9856, 19)
Unique studies: 58

Old split:
Split
train    7656
val      2200
Name: count, dtype: int64

5-FOLD TRIPLET MANIFEST
Shape: (9856, 20)
Missing fold assignments: 0
Unique studies: 58
✓ Every study belongs to exactly one fold.

Fold manifest summary:


,Fold,Studies,Series,Triplets
0,0,12,68,2194
1,1,12,72,2489
2,2,12,69,1627
3,3,11,58,1541
4,4,11,69,2005



TRIPLETS BY PLANE AND FOLD


Fold,0,1,2,3,4,Total
Anatomical_Plane,,,,,,
Axial,569,567,455,426,621,2638
Coronal,643,789,501,508,675,3116
Sagittal,982,1133,671,607,709,4102



LEAKAGE CHECK
Fold 0: train studies=46, val studies=12, overlap=0
Fold 1: train studies=46, val studies=12, overlap=0
Fold 2: train studies=46, val studies=12, overlap=0
Fold 3: train studies=47, val studies=11, overlap=0
Fold 4: train studies=47, val studies=11, overlap=0

✓ NO STUDY-LEVEL LEAKAGE ACROSS ANY FOLD

Saved:
/kaggle/working/rsna_score_improvement/splits/labelled_triplets_5fold.parquet
/kaggle/working/rsna_score_improvement/splits/fold_manifest_audit.csv


In [8]:
import json
from pathlib import Path
import torch

OLD_NOTEBOOK = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-supervised-fine-tuning/"
    "__notebook__.ipynb"
)

OLD_MODEL = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-supervised-fine-tuning/"
    "supervised/best_supervised_model.pt"
)

# ============================================================
# 1. Inspect saved checkpoint
# ============================================================

print("=" * 90)
print("1. BASELINE CHECKPOINT")
print("=" * 90)

checkpoint = torch.load(
    OLD_MODEL,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:")
print(type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    print(list(checkpoint.keys()))

    for key, value in checkpoint.items():

        if isinstance(value, (int, float, str, bool)):
            print(f"{key}: {value}")

        elif isinstance(value, dict):
            print(
                f"{key}: dict with "
                f"{len(value)} entries"
            )

# ============================================================
# 2. Inspect relevant code from old notebook
# ============================================================

print("\n" + "=" * 90)
print("2. RELEVANT BASELINE TRAINING CODE")
print("=" * 90)

with open(
    OLD_NOTEBOOK,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)

keywords = [
    "class ",
    "resnet",
    "encoder",
    "classifier",
    "BCEWithLogitsLoss",
    "pos_weight",
    "Adam",
    "AdamW",
    "optimizer",
    "scheduler",
    "encoder_lr",
    "classifier_lr",
    "freeze",
    "requires_grad",
    "epoch",
    "AUROC",
    "roc_auc",
    "aggregate",
    "groupby",
    "mean",
    "max",
]

matched = []

for i, cell in enumerate(nb["cells"]):

    if cell.get("cell_type") != "code":
        continue

    source = "".join(
        cell.get("source", [])
    )

    if any(
        keyword.lower() in source.lower()
        for keyword in keywords
    ):
        matched.append((i, source))

print(
    f"\nRelevant code cells found: "
    f"{len(matched)}"
)

for i, source in matched:

    print("\n" + "-" * 90)
    print(f"CELL {i}")
    print("-" * 90)

    # Avoid enormous output
    lines = source.splitlines()

    for line in lines[:80]:
        print(line)

    if len(lines) > 80:
        print(
            f"... + {len(lines) - 80} more lines"
        )

1. BASELINE CHECKPOINT

Checkpoint type:
<class 'dict'>

Checkpoint keys:
['epoch', 'model_state_dict', 'optimizer_state_dict', 'val_loss', 'label_cols']
epoch: 6
model_state_dict: dict with 122 entries
optimizer_state_dict: dict with 2 entries
val_loss: 0.8904742002487183

2. RELEVANT BASELINE TRAINING CODE

Relevant code cells found: 12

------------------------------------------------------------------------------------------
CELL 0
------------------------------------------------------------------------------------------
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch

# Competition data
COMP_ROOT = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

# Saved notebook outputs
NOTEBOOK_ROOT = Path("/kaggle/input/notebooks")

# Find SSL encoder only inside notebook inputs
encoder_files = list(
    NOTEBOOK_ROOT.rglob("best_encoder.pt")
)

assert len(encoder_files) == 1, (
    f"Expected 1 best_encoder.pt, found {len(enc

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

OUTPUT_ROOT = Path(
    "/kaggle/working/rsna_score_improvement"
)

WEIGHT_DIR = OUTPUT_ROOT / "class_weights"
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)

all_weight_rows = []

print("=" * 85)
print("5-FOLD CLASS WEIGHTS")
print("=" * 85)

for fold in range(5):

    train_studies = cv_df[
        cv_df["Fold"] != fold
    ].copy()

    val_studies = cv_df[
        cv_df["Fold"] == fold
    ].copy()

    positive = (
        train_studies[TARGETS]
        .sum()
        .astype(int)
    )

    negative = (
        len(train_studies) - positive
    )

    pos_weight = (
        negative / positive
    )

    fold_weights = pd.DataFrame({
        "Fold": fold,
        "Target": TARGETS,
        "Positive": positive.values,
        "Negative": negative.values,
        "PosWeight": pos_weight.values,
    })

    all_weight_rows.append(
        fold_weights
    )

    print(
        f"\nFold {fold} | "
        f"train={len(train_studies)} "
        f"val={len(val_studies)}"
    )

    display(
        fold_weights[
            ["Target",
             "Positive",
             "Negative",
             "PosWeight"]
        ].round(3)
    )

    fold_weights.to_csv(
        WEIGHT_DIR /
        f"fold_{fold}_class_weights.csv",
        index=False
    )

all_weights = pd.concat(
    all_weight_rows,
    ignore_index=True
)

all_weights.to_csv(
    WEIGHT_DIR /
    "all_fold_class_weights.csv",
    index=False
)

print("\n" + "=" * 85)
print("WEIGHT RANGE CHECK")
print("=" * 85)

summary = (
    all_weights
    .groupby("Target")["PosWeight"]
    .agg(["min", "max", "mean"])
    .round(3)
)

display(summary)

print("\nSaved to:")
print(WEIGHT_DIR)

5-FOLD CLASS WEIGHTS

Fold 0 | train=46 val=12


,Target,Positive,Negative,PosWeight
0,ACL,18,28,1.556
1,MCL,6,40,6.667
2,Medial Meniscus,21,25,1.190
3,Lateral Meniscus,19,27,1.421
4,Medial OA,13,33,2.538
5,Lateral OA,8,38,4.750
6,PF OA,16,30,1.875
7,Effusion,27,19,0.704
8,Synovitis,20,26,1.300
9,Baker's,10,36,3.600



Fold 1 | train=46 val=12


,Target,Positive,Negative,PosWeight
0,ACL,19,27,1.421
1,MCL,7,39,5.571
2,Medial Meniscus,19,27,1.421
3,Lateral Meniscus,19,27,1.421
4,Medial OA,11,35,3.182
5,Lateral OA,9,37,4.111
6,PF OA,16,30,1.875
7,Effusion,28,18,0.643
8,Synovitis,22,24,1.091
9,Baker's,9,37,4.111



Fold 2 | train=46 val=12


...


Fold 4 | train=47 val=11


,Target,Positive,Negative,PosWeight
0,ACL,21,26,1.238
1,MCL,8,39,4.875
2,Medial Meniscus,22,25,1.136
3,Lateral Meniscus,17,30,1.765
4,Medial OA,12,35,2.917
5,Lateral OA,9,38,4.222
6,PF OA,17,30,1.765
7,Effusion,29,18,0.621
8,Synovitis,22,25,1.136
9,Baker's,9,38,4.222



WEIGHT RANGE CHECK


,min,max,mean
Target,,,
ACL,1.238,1.556,1.424
Baker's,3.600,4.222,3.847
Contusion,1.875,2.357,2.061
Effusion,0.621,0.704,0.658
Fracture,2.067,2.357,2.226
Lateral Meniscus,1.421,1.765,1.528
Lateral OA,3.600,4.875,4.312
MCL,4.750,6.667,5.515
Medial Meniscus,1.091,1.421,1.238



Saved to:
/kaggle/working/rsna_score_improvement/class_weights


In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# Paths
# ============================================================

SUP_DIR = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-supervised-fine-tuning/supervised"
)

SSL_DIR = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-clean-ssl-training/ssl_training"
)

CV_MANIFEST_PATH = Path(
    "/kaggle/working/rsna_score_improvement/"
    "splits/labelled_triplets_5fold.parquet"
)

BEST_ENCODER_PATH = SSL_DIR / "best_encoder.pt"

# ============================================================
# Load reusable files
# ============================================================

cv_manifest = pd.read_parquet(
    CV_MANIFEST_PATH
)

percentiles = pd.read_csv(
    SUP_DIR / "labelled_series_percentiles.csv"
)

# ============================================================
# 1. Core files
# ============================================================

print("=" * 90)
print("V2 TRAINING PREFLIGHT")
print("=" * 90)

print("\nBest SSL encoder:")
print(BEST_ENCODER_PATH)
print("Exists:", BEST_ENCODER_PATH.exists())

print("\nCV manifest:")
print(CV_MANIFEST_PATH)
print("Exists:", CV_MANIFEST_PATH.exists())

# ============================================================
# 2. Manifest structure
# ============================================================

print("\n" + "=" * 90)
print("CV MANIFEST")
print("=" * 90)

print("Shape:", cv_manifest.shape)
print("Studies:", cv_manifest["StudyInstanceUID"].nunique())
print("Series:", cv_manifest["SeriesInstanceUID"].nunique())

print("\nColumns:")
print(cv_manifest.columns.tolist())

print("\nFolds:")
print(
    cv_manifest["Fold"]
    .value_counts()
    .sort_index()
)

# ============================================================
# 3. Percentile cache
# ============================================================

print("\n" + "=" * 90)
print("LABELLED SERIES PERCENTILES")
print("=" * 90)

print("Shape:", percentiles.shape)

print("\nColumns:")
print(percentiles.columns.tolist())

display(percentiles.head())

# ============================================================
# 4. Check a sample of MRI paths
# ============================================================

print("\n" + "=" * 90)
print("MRI PATH CHECK")
print("=" * 90)

path_cols = [
    "PreviousPath",
    "CentrePath",
    "NextPath",
]

rng = np.random.default_rng(42)

sample_n = min(
    100,
    len(cv_manifest)
)

sample_idx = rng.choice(
    len(cv_manifest),
    size=sample_n,
    replace=False
)

sample = cv_manifest.iloc[sample_idx]

for col in path_cols:

    exists = sample[col].map(
        lambda x: Path(x).exists()
    )

    print(
        f"{col:<15}: "
        f"{exists.sum()}/{len(exists)} exist"
    )

# ============================================================
# 5. Plane distribution
# ============================================================

print("\n" + "=" * 90)
print("MRI PLANES")
print("=" * 90)

print(
    cv_manifest["Anatomical_Plane"]
    .value_counts()
)

# ============================================================
# 6. Fold-specific weights
# ============================================================

print("\n" + "=" * 90)
print("CLASS-WEIGHT FILES")
print("=" * 90)

WEIGHT_DIR = Path(
    "/kaggle/working/rsna_score_improvement/"
    "class_weights"
)

for fold in range(5):

    p = WEIGHT_DIR / f"fold_{fold}_class_weights.csv"

    print(
        f"Fold {fold}: "
        f"{'✓' if p.exists() else 'MISSING'}"
    )

print("\n" + "=" * 90)
print("PREFLIGHT COMPLETE")
print("=" * 90)

V2 TRAINING PREFLIGHT

Best SSL encoder:
/kaggle/input/notebooks/elliotyang37/rsna-knee-mri-clean-ssl-training/ssl_training/best_encoder.pt
Exists: True

CV manifest:
/kaggle/working/rsna_score_improvement/splits/labelled_triplets_5fold.parquet
Exists: True

CV MANIFEST
Shape: (9856, 20)
Studies: 58
Series: 336

Columns:
['StudyInstanceUID', 'SeriesInstanceUID', 'Anatomical_Plane', 'PreviousPath', 'CentrePath', 'NextPath', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture', 'BaselineSplit', 'Fold']

Folds:
Fold
0    2194
1    2489
2    1627
3    1541
4    2005
Name: count, dtype: int64

LABELLED SERIES PERCENTILES
Shape: (336, 6)

Columns:
['StudyInstanceUID', 'SeriesInstanceUID', 'Anatomical_Plane', 'P01', 'P99', 'Valid']


,StudyInstanceUID,SeriesInstanceUID,Anatomical_Plane,P01,P99,Valid
0,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.10791858932231443712...,Sagittal,0.0,601.0,True
1,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.12451507871565834868...,Coronal,0.0,888.0,True
2,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.21497837050858589852...,Coronal,0.0,378.0,True
3,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.37372672584387115708...,Sagittal,0.0,398.0,True
4,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.67495435033389756963...,Axial,0.0,351.0,True



MRI PATH CHECK
PreviousPath   : 100/100 exist
CentrePath     : 100/100 exist
NextPath       : 100/100 exist

MRI PLANES
Anatomical_Plane
Sagittal    4102
Coronal     3116
Axial       2638
Name: count, dtype: int64

CLASS-WEIGHT FILES
Fold 0: ✓
Fold 1: ✓
Fold 2: ✓
Fold 3: ✓
Fold 4: ✓

PREFLIGHT COMPLETE


In [11]:
# ============================================================
# STEP 2C — V2 MRI Dataset
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import pydicom

import torch
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)

TARGET_SIZE = 224
TARGET_SPACING = 0.5

LABEL_COLS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]


# ============================================================
# Percentile lookup
# ============================================================

percentile_lookup = {}

for row in percentiles.itertuples(index=False):

    key = (
        str(row.StudyInstanceUID),
        str(row.SeriesInstanceUID),
    )

    percentile_lookup[key] = (
        float(row.P01),
        float(row.P99),
    )

print(
    "Percentile lookup entries:",
    len(percentile_lookup)
)


# ============================================================
# DICOM loading
# ============================================================

def read_dicom_for_model(path):

    ds = pydicom.dcmread(
        path,
        force=True
    )

    image = ds.pixel_array.astype(
        np.float32
    )

    slope = float(
        getattr(ds, "RescaleSlope", 1.0)
    )

    intercept = float(
        getattr(ds, "RescaleIntercept", 0.0)
    )

    image = image * slope + intercept

    try:

        spacing = ds.PixelSpacing

        row_spacing = float(spacing[0])
        col_spacing = float(spacing[1])

    except Exception:

        row_spacing = TARGET_SPACING
        col_spacing = TARGET_SPACING

    return (
        image,
        row_spacing,
        col_spacing,
    )


# ============================================================
# Intensity normalization
# ============================================================

def normalize_image(
    image,
    p01,
    p99
):

    image = np.clip(
        image,
        p01,
        p99
    )

    image = (
        (image - p01)
        / (p99 - p01 + 1e-6)
    )

    return image.astype(
        np.float32
    )


# ============================================================
# Spacing-aware resize
# ============================================================

def resize_to_spacing(
    image,
    row_spacing,
    col_spacing
):

    h, w = image.shape

    new_h = max(
        1,
        int(round(
            h * row_spacing
            / TARGET_SPACING
        ))
    )

    new_w = max(
        1,
        int(round(
            w * col_spacing
            / TARGET_SPACING
        ))
    )

    x = torch.from_numpy(
        image
    ).float()

    x = x[None, None]

    x = F.interpolate(
        x,
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    )

    return x[0, 0]


# ============================================================
# Centre crop / pad
# ============================================================

def centre_crop_pad(
    image,
    target_size=224
):

    h, w = image.shape

    # --------------------------
    # Centre crop
    # --------------------------

    if h > target_size:

        start = (
            h - target_size
        ) // 2

        image = image[
            start:start + target_size,
            :
        ]

    if w > target_size:

        start = (
            w - target_size
        ) // 2

        image = image[
            :,
            start:start + target_size
        ]

    # --------------------------
    # Padding
    # --------------------------

    h, w = image.shape

    pad_h = max(
        0,
        target_size - h
    )

    pad_w = max(
        0,
        target_size - w
    )

    pad_top = pad_h // 2
    pad_bottom = (
        pad_h - pad_top
    )

    pad_left = pad_w // 2
    pad_right = (
        pad_w - pad_left
    )

    image = F.pad(
        image,
        (
            pad_left,
            pad_right,
            pad_top,
            pad_bottom,
        ),
        value=0
    )

    return image


# ============================================================
# Process one MRI slice
# ============================================================

def process_slice(
    path,
    p01,
    p99
):

    image, rs, cs = (
        read_dicom_for_model(path)
    )

    image = normalize_image(
        image,
        p01,
        p99
    )

    image = resize_to_spacing(
        image,
        rs,
        cs
    )

    image = centre_crop_pad(
        image,
        TARGET_SIZE
    )

    return image


# ============================================================
# MRI-safe training augmentation
#
# No horizontal flipping:
# medial/lateral labels make anatomical flips risky.
# ============================================================

def intensity_augment(x):

    # Mild contrast
    scale = (
        0.90
        + 0.20 * torch.rand(1)
    )

    # Mild brightness
    shift = (
        -0.04
        + 0.08 * torch.rand(1)
    )

    x = x * scale + shift

    # Small Gaussian MRI noise
    if torch.rand(1).item() < 0.5:

        noise = (
            torch.randn_like(x)
            * 0.015
        )

        x = x + noise

    return torch.clamp(
        x,
        0.0,
        1.0
    )


# ============================================================
# Dataset
# ============================================================

class KneeV2Dataset(Dataset):

    def __init__(
        self,
        manifest,
        percentile_lookup,
        augment=False
    ):

        self.df = (
            manifest
            .reset_index(drop=True)
            .copy()
        )

        self.percentile_lookup = (
            percentile_lookup
        )

        self.augment = augment

    def __len__(self):

        return len(self.df)

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        study_uid = str(
            row["StudyInstanceUID"]
        )

        series_uid = str(
            row["SeriesInstanceUID"]
        )

        key = (
            study_uid,
            series_uid
        )

        if key not in self.percentile_lookup:

            raise KeyError(
                f"No percentile cache for {key}"
            )

        p01, p99 = (
            self.percentile_lookup[key]
        )

        previous = process_slice(
            row["PreviousPath"],
            p01,
            p99
        )

        centre = process_slice(
            row["CentrePath"],
            p01,
            p99
        )

        next_slice = process_slice(
            row["NextPath"],
            p01,
            p99
        )

        # 2.5D:
        # previous / centre / next
        image = torch.stack(
            [
                previous,
                centre,
                next_slice
            ],
            dim=0
        )

        if self.augment:

            image = intensity_augment(
                image
            )

        labels = torch.tensor(
            row[LABEL_COLS]
            .to_numpy(
                dtype=np.float32
            ),
            dtype=torch.float32
        )

        return {
            "image": image,
            "labels": labels,

            "study_uid":
                study_uid,

            "series_uid":
                series_uid,

            "plane":
                str(
                    row["Anatomical_Plane"]
                ),

            "manifest_index":
                int(idx),
        }


print("✓ KneeV2Dataset created")

Percentile lookup entries: 336
✓ KneeV2Dataset created


In [12]:
# ============================================================
# STEP 2D — FOLD 0 CPU SMOKE TEST
# ============================================================

TEST_FOLD = 0

train_df = cv_manifest[
    cv_manifest["Fold"] != TEST_FOLD
].copy()

val_df = cv_manifest[
    cv_manifest["Fold"] == TEST_FOLD
].copy()

print("=" * 80)
print("FOLD 0")
print("=" * 80)

print(
    "Train studies:",
    train_df["StudyInstanceUID"]
    .nunique()
)

print(
    "Validation studies:",
    val_df["StudyInstanceUID"]
    .nunique()
)

print(
    "Train triplets:",
    len(train_df)
)

print(
    "Validation triplets:",
    len(val_df)
)


# ============================================================
# Datasets
# ============================================================

train_dataset = KneeV2Dataset(
    train_df,
    percentile_lookup,
    augment=True
)

val_dataset = KneeV2Dataset(
    val_df,
    percentile_lookup,
    augment=False
)


# ============================================================
# Study-balanced sampling
#
# Without this, a study with many MRI triplets contributes
# much more to the loss than a study with fewer triplets.
# ============================================================

study_counts = (
    train_df[
        "StudyInstanceUID"
    ]
    .value_counts()
)

sample_weights = (
    train_df[
        "StudyInstanceUID"
    ]
    .map(
        lambda uid:
            1.0 / study_counts[uid]
    )
    .to_numpy(
        dtype=np.float64
    )
)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_df),
    replacement=True
)


# ============================================================
# Tiny CPU loaders
# ============================================================

train_loader_test = DataLoader(
    train_dataset,
    batch_size=4,
    sampler=sampler,
    num_workers=2,
    pin_memory=False
)

val_loader_test = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=False
)


# ============================================================
# Read one batch
# ============================================================

batch = next(
    iter(train_loader_test)
)

print("\n" + "=" * 80)
print("BATCH CHECK")
print("=" * 80)

print(
    "Image shape:",
    batch["image"].shape
)

print(
    "Label shape:",
    batch["labels"].shape
)

print(
    "Image min:",
    batch["image"].min().item()
)

print(
    "Image max:",
    batch["image"].max().item()
)

print(
    "Image mean:",
    batch["image"].mean().item()
)

print(
    "Planes:",
    batch["plane"]
)

print(
    "Study UIDs:",
    len(batch["study_uid"])
)

assert (
    batch["image"].shape
    == (4, 3, 224, 224)
)

assert (
    batch["labels"].shape
    == (4, 12)
)

assert torch.isfinite(
    batch["image"]
).all()

assert torch.isfinite(
    batch["labels"]
).all()

print(
    "\n✓ V2 DATASET + LOADER "
    "SMOKE TEST PASSED"
)

FOLD 0
Train studies: 46
Validation studies: 12
Train triplets: 7662
Validation triplets: 2194

BATCH CHECK
Image shape: torch.Size([4, 3, 224, 224])
Label shape: torch.Size([4, 12])
Image min: 0.0
Image max: 1.0
Image mean: 0.4246242344379425
Planes: ['Axial', 'Sagittal', 'Sagittal', 'Axial']
Study UIDs: 4

✓ V2 DATASET + LOADER SMOKE TEST PASSED


In [21]:
# ============================================================
# STEP 3B — BUILD 58-STUDY OOF PREDICTIONS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

MODEL_ROOT = Path(
    "/kaggle/working/rsna_score_improvement/models"
)

OOF_DIR = Path(
    "/kaggle/working/rsna_score_improvement/oof"
)

OOF_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# Load best study-level predictions from every fold
# ============================================================

fold_predictions = []

for fold in range(5):

    path = (
        MODEL_ROOT /
        f"fold_{fold}" /
        "best_val_study_predictions.csv"
    )

    assert path.exists(), (
        f"Missing Fold {fold}: {path}"
    )

    df = pd.read_csv(path)

    df["Fold"] = fold

    fold_predictions.append(df)

    print(
        f"Fold {fold}: "
        f"{len(df)} studies loaded"
    )

# ============================================================
# Combine
# ============================================================

oof = pd.concat(
    fold_predictions,
    ignore_index=True
)

print("\n" + "=" * 90)
print("OOF SAFETY CHECK")
print("=" * 90)

print("Rows:", len(oof))
print(
    "Unique studies:",
    oof["StudyInstanceUID"].nunique()
)

print("\nStudies per fold:")
print(
    oof["Fold"]
    .value_counts()
    .sort_index()
)

# Exactly 58 predictions
assert len(oof) == 58

# Each study must appear exactly once
assert (
    oof["StudyInstanceUID"]
    .nunique()
    == 58
)

duplicates = (
    oof["StudyInstanceUID"]
    .duplicated()
    .sum()
)

assert duplicates == 0

print(
    "\n✓ Every labelled study has "
    "exactly one OOF prediction"
)

# ============================================================
# Calculate per-target AUROC
# ============================================================

metric_rows = []

for target in LABEL_COLS:

    y_true = oof[
        f"{target}_true"
    ].values

    y_prob = oof[
        f"{target}_prob"
    ].values

    auc = roc_auc_score(
        y_true,
        y_prob
    )

    metric_rows.append({
        "Target": target,
        "AUROC": auc
    })

metrics_df = pd.DataFrame(
    metric_rows
)

macro_auc = (
    metrics_df["AUROC"]
    .mean()
)

# ============================================================
# Results
# ============================================================

print("\n" + "=" * 90)
print("58-STUDY OOF RESULTS")
print("=" * 90)

display(
    metrics_df.round(4)
)

print(
    "\nOOF MACRO AUROC:",
    round(macro_auc, 4)
)

# ============================================================
# Extra fold information
# ============================================================

fold_summary = pd.DataFrame({
    "Fold": [0, 1, 2, 3, 4],
    "Best_Fold_AUROC": [
        0.5919,
        0.6030,
        0.4554,
        0.6152,
        0.5928,
    ],
})

print("\nFold checkpoint scores:")
display(fold_summary)

# ============================================================
# Save
# ============================================================

oof.to_csv(
    OOF_DIR /
    "oof_study_predictions.csv",
    index=False
)

metrics_df.to_csv(
    OOF_DIR /
    "oof_target_metrics.csv",
    index=False
)

pd.DataFrame({
    "Metric": ["Macro_AUROC"],
    "Value": [macro_auc]
}).to_csv(
    OOF_DIR /
    "oof_summary.csv",
    index=False
)

print("\nSaved:")
print(
    OOF_DIR /
    "oof_study_predictions.csv"
)
print(
    OOF_DIR /
    "oof_target_metrics.csv"
)
print(
    OOF_DIR /
    "oof_summary.csv"
)

Fold 0: 12 studies loaded
Fold 1: 12 studies loaded
Fold 2: 12 studies loaded
Fold 3: 11 studies loaded
Fold 4: 11 studies loaded

OOF SAFETY CHECK
Rows: 58
Unique studies: 58

Studies per fold:
Fold
0    12
1    12
2    12
3    11
4    11
Name: count, dtype: int64

✓ Every labelled study has exactly one OOF prediction

58-STUDY OOF RESULTS


,Target,AUROC
0,ACL,0.5760
1,MCL,0.6213
2,Medial Meniscus,0.5685
3,Lateral Meniscus,0.4919
4,Medial OA,0.5736
5,Lateral OA,0.5145
6,PF OA,0.5959
7,Effusion,0.6360
8,Synovitis,0.4970
9,Baker's,0.4004



OOF MACRO AUROC: 0.5432

Fold checkpoint scores:


,Fold,Best_Fold_AUROC
0,0,0.5919
1,1,0.6030
2,2,0.4554
3,3,0.6152
4,4,0.5928



Saved:
/kaggle/working/rsna_score_improvement/oof/oof_study_predictions.csv
/kaggle/working/rsna_score_improvement/oof/oof_target_metrics.csv
/kaggle/working/rsna_score_improvement/oof/oof_summary.csv


In [23]:
# ============================================================
# STEP 4A — ORIGINAL-STYLE BASELINE CV TRAINER
# ============================================================

from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


def train_baseline_fold(
    fold,
    batch_size=64
):

    print("\n" + "=" * 90)
    print(f"BASELINE CV — FOLD {fold}")
    print("=" * 90)

    fold_dir = Path(
        f"/kaggle/working/"
        f"rsna_score_improvement/"
        f"baseline_cv/fold_{fold}"
    )

    fold_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # ========================================================
    # Split
    # ========================================================

    train_df = cv_manifest[
        cv_manifest["Fold"] != fold
    ].copy()

    val_df = cv_manifest[
        cv_manifest["Fold"] == fold
    ].copy()

    print(
        "Train studies:",
        train_df["StudyInstanceUID"].nunique()
    )

    print(
        "Validation studies:",
        val_df["StudyInstanceUID"].nunique()
    )

    print(
        "Train triplets:",
        len(train_df)
    )

    print(
        "Validation triplets:",
        len(val_df)
    )

    # ========================================================
    # Dataset
    #
    # Baseline-style:
    # no new V2 intensity augmentation
    # ========================================================

    train_dataset = KneeV2Dataset(
        train_df,
        percentile_lookup,
        augment=False
    )

    val_dataset = KneeV2Dataset(
        val_df,
        percentile_lookup,
        augment=False
    )

    # ========================================================
    # Loader
    #
    # Baseline-style:
    # normal shuffle rather than study-balanced sampling
    # ========================================================

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True
    )

    # ========================================================
    # Fold-specific class weights
    # ========================================================

    weight_df = pd.read_csv(
        WEIGHT_DIR /
        f"fold_{fold}_class_weights.csv"
    )

    weight_df = (
        weight_df
        .set_index("Target")
        .loc[LABEL_COLS]
    )

    pos_weight = torch.tensor(
        weight_df["PosWeight"].values,
        dtype=torch.float32,
        device=device
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    # ========================================================
    # Original-style model
    # ========================================================

    set_seed(
        2000 + fold
    )

    encoder = resnet18(
        weights=None
    )

    encoder.fc = nn.Identity()

    ssl_state = torch.load(
        BEST_ENCODER_PATH,
        map_location="cpu",
        weights_only=True
    )

    encoder.load_state_dict(
        ssl_state,
        strict=True
    )

    model = KneeClassifierV2(
        encoder=encoder,
        num_labels=len(LABEL_COLS),
        dropout=0.30
    ).to(device)

    # Everything trainable from Epoch 1
    for param in model.parameters():
        param.requires_grad = True

    # ========================================================
    # Optimizer — original recipe
    # ========================================================

    optimizer = torch.optim.AdamW(
        [
            {
                "params":
                    model.encoder.parameters(),
                "lr": 1e-5
            },
            {
                "params":
                    model.classifier.parameters(),
                "lr": 1e-4
            }
        ],
        weight_decay=1e-4
    )

    EPOCHS = 10

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=EPOCHS
        )
    )

    use_amp = (
        device.type == "cuda"
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )

    # ========================================================
    # We will save TWO best versions:
    #
    # A) best macro AUROC
    # B) best validation loss
    #
    # One training run gives us both comparisons.
    # ========================================================

    best_auc = -np.inf
    best_loss = np.inf

    history = []

    # ========================================================
    # Train
    # ========================================================

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        start = time.time()

        model.train()

        running_loss = 0.0
        seen = 0

        progress = tqdm(
            train_loader,
            desc=(
                f"Baseline Fold {fold} "
                f"Epoch {epoch}/{EPOCHS}"
            )
        )

        for batch in progress:

            images = batch["image"].to(
                device,
                non_blocking=True
            )

            labels = batch["labels"].to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.amp.autocast(
                device_type=device.type,
                enabled=use_amp
            ):

                logits = model(images)

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

            bs = images.size(0)

            running_loss += (
                loss.item() * bs
            )

            seen += bs

        train_loss = (
            running_loss / seen
        )

        # ====================================================
        # Validation predictions
        # ====================================================

        triplet_preds = (
            predict_validation_triplets(
                model,
                val_loader
            )
        )

        study_preds = (
            aggregate_mean_logit(
                triplet_preds
            )
        )

        macro_auc, target_auc = (
            calculate_macro_auc(
                study_preds
            )
        )

        # ====================================================
        # Study-level weighted BCE validation loss
        # ====================================================

        study_logits = []
        study_labels = []

        for uid, g in (
            triplet_preds
            .groupby("StudyInstanceUID")
        ):

            logits = np.array([
                g[
                    f"{target}_logit"
                ].mean()
                for target in LABEL_COLS
            ])

            labels = np.array([
                g[
                    f"{target}_true"
                ].iloc[0]
                for target in LABEL_COLS
            ])

            study_logits.append(
                logits
            )

            study_labels.append(
                labels
            )

        study_logits = torch.tensor(
            np.vstack(study_logits),
            dtype=torch.float32,
            device=device
        )

        study_labels = torch.tensor(
            np.vstack(study_labels),
            dtype=torch.float32,
            device=device
        )

        val_loss = criterion(
            study_logits,
            study_labels
        ).item()

        # ====================================================
        # Save best AUROC version
        # ====================================================

        if macro_auc > best_auc:

            best_auc = macro_auc

            torch.save(
                {
                    "fold": fold,
                    "epoch": epoch,
                    "macro_auc": macro_auc,
                    "val_loss": val_loss,
                    "model_state_dict":
                        model.state_dict(),
                    "label_cols": LABEL_COLS
                },
                fold_dir /
                "best_by_auc_model.pt"
            )

            triplet_preds.to_parquet(
                fold_dir /
                "best_by_auc_triplets.parquet",
                index=False
            )

            study_preds.to_csv(
                fold_dir /
                "best_by_auc_studies.csv",
                index=False
            )

        # ====================================================
        # Save best validation-loss version
        # ====================================================

        if val_loss < best_loss:

            best_loss = val_loss

            torch.save(
                {
                    "fold": fold,
                    "epoch": epoch,
                    "macro_auc": macro_auc,
                    "val_loss": val_loss,
                    "model_state_dict":
                        model.state_dict(),
                    "label_cols": LABEL_COLS
                },
                fold_dir /
                "best_by_loss_model.pt"
            )

            triplet_preds.to_parquet(
                fold_dir /
                "best_by_loss_triplets.parquet",
                index=False
            )

            study_preds.to_csv(
                fold_dir /
                "best_by_loss_studies.csv",
                index=False
            )

        scheduler.step()

        elapsed = (
            time.time() - start
        )

        history.append({
            "fold": fold,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "macro_auc": macro_auc,
            "best_auc": best_auc,
            "best_loss": best_loss,
            "minutes": elapsed / 60,
        })

        pd.DataFrame(
            history
        ).to_csv(
            fold_dir /
            "training_history.csv",
            index=False
        )

        print(
            f"\nEpoch {epoch}/{EPOCHS}"
        )

        print(
            f"Train loss:  {train_loss:.4f}"
        )

        print(
            f"Val loss:    {val_loss:.4f}"
        )

        print(
            f"Macro AUROC: {macro_auc:.4f}"
        )

        print(
            f"Best AUROC:  {best_auc:.4f}"
        )

        print(
            f"Best loss:   {best_loss:.4f}"
        )

        print(
            f"Time:        {elapsed/60:.1f} min"
        )

    print(
        "\n" + "=" * 90
    )

    print(
        f"BASELINE FOLD {fold} COMPLETE"
    )

    print(
        "=" * 90
    )

    print(
        "Best AUROC:",
        round(best_auc, 4)
    )

    print(
        "Best validation loss:",
        round(best_loss, 4)
    )

    del model
    torch.cuda.empty_cache()

    return best_auc, best_loss

In [24]:
baseline_fold0_auc, baseline_fold0_loss = (
    train_baseline_fold(
        fold=0,
        batch_size=64
    )
)


BASELINE CV — FOLD 0
Train studies: 46
Validation studies: 12
Train triplets: 7662
Validation triplets: 2194


Baseline Fold 0 Epoch 1/10:   0%|          | 0/120 [00:00<?, ?it/s]


Epoch 1/10
Train loss:  0.9548
Val loss:    0.9574
Macro AUROC: 0.5297
Best AUROC:  0.5297
Best loss:   0.9574
Time:        0.9 min


Baseline Fold 0 Epoch 2/10:   0%|          | 0/120 [00:00<?, ?it/s]


Epoch 2/10
Train loss:  0.9129
Val loss:    0.9430
Macro AUROC: 0.6375
Best AUROC:  0.6375
Best loss:   0.9430
Time:        0.9 min


...


Epoch 8/10
Train loss:  0.8085
Val loss:    0.9472
Macro AUROC: 0.6845
Best AUROC:  0.6905
Best loss:   0.9368
Time:        0.9 min


Baseline Fold 0 Epoch 9/10:   0%|          | 0/120 [00:00<?, ?it/s]


Epoch 9/10
Train loss:  0.8079
Val loss:    0.9459
Macro AUROC: 0.6921
Best AUROC:  0.6921
Best loss:   0.9368
Time:        0.9 min


Baseline Fold 0 Epoch 10/10:   0%|          | 0/120 [00:00<?, ?it/s]


Epoch 10/10
Train loss:  0.8038
Val loss:    0.9468
Macro AUROC: 0.6892
Best AUROC:  0.6921
Best loss:   0.9368
Time:        0.9 min

BASELINE FOLD 0 COMPLETE
Best AUROC: 0.6921
Best validation loss: 0.9368


In [25]:
history0 = pd.read_csv(
    "/kaggle/working/rsna_score_improvement/"
    "baseline_cv/fold_0/training_history.csv"
)

print("Best AUROC epoch:")
display(
    history0.loc[
        history0["macro_auc"].idxmax()
    ]
)

print("\nBest validation-loss epoch:")
display(
    history0.loc[
        history0["val_loss"].idxmin()
    ]
)

Best AUROC epoch:


fold          0.000000
epoch         9.000000
train_loss    0.807898
val_loss      0.945890
macro_auc     0.692110
best_auc      0.692110
best_loss     0.936823
minutes       0.867253
Name: 8, dtype: float64


Best validation-loss epoch:


fold          0.000000
epoch         5.000000
train_loss    0.841702
val_loss      0.936823
macro_auc     0.680710
best_auc      0.690482
best_loss     0.936823
minutes       0.866164
Name: 4, dtype: float64

In [27]:
baseline_fold1_auc, baseline_fold1_loss = (
    train_baseline_fold(
        fold=1,
        batch_size=64
    )
)


BASELINE CV — FOLD 1
Train studies: 46
Validation studies: 12
Train triplets: 7367
Validation triplets: 2489


Baseline Fold 1 Epoch 1/10:   0%|          | 0/116 [00:00<?, ?it/s]


Epoch 1/10
Train loss:  0.9597
Val loss:    0.9351
Macro AUROC: 0.6046
Best AUROC:  0.6046
Best loss:   0.9351
Time:        0.9 min


Baseline Fold 1 Epoch 2/10:   0%|          | 0/116 [00:00<?, ?it/s]


Epoch 2/10
Train loss:  0.9124
Val loss:    0.9290
Macro AUROC: 0.6472
Best AUROC:  0.6472
Best loss:   0.9290
Time:        0.9 min


...


Epoch 8/10
Train loss:  0.8298
Val loss:    0.9295
Macro AUROC: 0.6262
Best AUROC:  0.6624
Best loss:   0.9216
Time:        0.9 min


Baseline Fold 1 Epoch 9/10:   0%|          | 0/116 [00:00<?, ?it/s]


Epoch 9/10
Train loss:  0.8275
Val loss:    0.9214
Macro AUROC: 0.6252
Best AUROC:  0.6624
Best loss:   0.9214
Time:        0.9 min


Baseline Fold 1 Epoch 10/10:   0%|          | 0/116 [00:00<?, ?it/s]


Epoch 10/10
Train loss:  0.8254
Val loss:    0.9242
Macro AUROC: 0.6307
Best AUROC:  0.6624
Best loss:   0.9214
Time:        0.9 min

BASELINE FOLD 1 COMPLETE
Best AUROC: 0.6624
Best validation loss: 0.9214


In [28]:
baseline_fold2_auc, baseline_fold2_loss = (
    train_baseline_fold(
        fold=2,
        batch_size=64
    )
)


BASELINE CV — FOLD 2
Train studies: 46
Validation studies: 12
Train triplets: 8229
Validation triplets: 1627


Baseline Fold 2 Epoch 1/10:   0%|          | 0/129 [00:00<?, ?it/s]


Epoch 1/10
Train loss:  0.9104
Val loss:    0.8445
Macro AUROC: 0.4447
Best AUROC:  0.4447
Best loss:   0.8445
Time:        0.9 min


Baseline Fold 2 Epoch 2/10:   0%|          | 0/129 [00:00<?, ?it/s]


Epoch 2/10
Train loss:  0.8466
Val loss:    0.8578
Macro AUROC: 0.4148
Best AUROC:  0.4447
Best loss:   0.8445
Time:        0.9 min


...


Epoch 8/10
Train loss:  0.7472
Val loss:    0.8796
Macro AUROC: 0.4440
Best AUROC:  0.4594
Best loss:   0.8445
Time:        0.9 min


Baseline Fold 2 Epoch 9/10:   0%|          | 0/129 [00:00<?, ?it/s]


Epoch 9/10
Train loss:  0.7434
Val loss:    0.8790
Macro AUROC: 0.4440
Best AUROC:  0.4594
Best loss:   0.8445
Time:        0.9 min


Baseline Fold 2 Epoch 10/10:   0%|          | 0/129 [00:00<?, ?it/s]


Epoch 10/10
Train loss:  0.7433
Val loss:    0.8787
Macro AUROC: 0.4440
Best AUROC:  0.4594
Best loss:   0.8445
Time:        0.9 min

BASELINE FOLD 2 COMPLETE
Best AUROC: 0.4594
Best validation loss: 0.8445


In [30]:
baseline_fold3_auc, baseline_fold3_loss = (
    train_baseline_fold(
        fold=3,
        batch_size=64
    )
)


BASELINE CV — FOLD 3
Train studies: 47
Validation studies: 11
Train triplets: 8315
Validation triplets: 1541


Baseline Fold 3 Epoch 1/10:   0%|          | 0/130 [00:00<?, ?it/s]


Epoch 1/10
Train loss:  0.9374
Val loss:    0.9528
Macro AUROC: 0.5589
Best AUROC:  0.5589
Best loss:   0.9528
Time:        0.9 min


Baseline Fold 3 Epoch 2/10:   0%|          | 0/130 [00:00<?, ?it/s]


Epoch 2/10
Train loss:  0.8863
Val loss:    0.9411
Macro AUROC: 0.6636
Best AUROC:  0.6636
Best loss:   0.9411
Time:        0.9 min


...


Epoch 8/10
Train loss:  0.7847
Val loss:    0.9083
Macro AUROC: 0.6915
Best AUROC:  0.6915
Best loss:   0.9083
Time:        0.9 min


Baseline Fold 3 Epoch 9/10:   0%|          | 0/130 [00:00<?, ?it/s]


Epoch 9/10
Train loss:  0.7834
Val loss:    0.9084
Macro AUROC: 0.6915
Best AUROC:  0.6915
Best loss:   0.9083
Time:        0.9 min


Baseline Fold 3 Epoch 10/10:   0%|          | 0/130 [00:00<?, ?it/s]


Epoch 10/10
Train loss:  0.7797
Val loss:    0.9090
Macro AUROC: 0.6915
Best AUROC:  0.6915
Best loss:   0.9083
Time:        0.9 min

BASELINE FOLD 3 COMPLETE
Best AUROC: 0.6915
Best validation loss: 0.9083


In [31]:
baseline_fold4_auc, baseline_fold4_loss = (
    train_baseline_fold(
        fold=4,
        batch_size=64
    )
)


BASELINE CV — FOLD 4
Train studies: 47
Validation studies: 11
Train triplets: 7851
Validation triplets: 2005


Baseline Fold 4 Epoch 1/10:   0%|          | 0/123 [00:00<?, ?it/s]


Epoch 1/10
Train loss:  0.9230
Val loss:    0.9069
Macro AUROC: 0.5274
Best AUROC:  0.5274
Best loss:   0.9069
Time:        0.9 min


Baseline Fold 4 Epoch 2/10:   0%|          | 0/123 [00:00<?, ?it/s]


Epoch 2/10
Train loss:  0.8685
Val loss:    0.9037
Macro AUROC: 0.5607
Best AUROC:  0.5607
Best loss:   0.9037
Time:        0.9 min


...


Epoch 8/10
Train loss:  0.7834
Val loss:    0.9151
Macro AUROC: 0.5960
Best AUROC:  0.6011
Best loss:   0.9030
Time:        0.9 min


Baseline Fold 4 Epoch 9/10:   0%|          | 0/123 [00:00<?, ?it/s]


Epoch 9/10
Train loss:  0.7801
Val loss:    0.9165
Macro AUROC: 0.5849
Best AUROC:  0.6011
Best loss:   0.9030
Time:        0.9 min


Baseline Fold 4 Epoch 10/10:   0%|          | 0/123 [00:00<?, ?it/s]


Epoch 10/10
Train loss:  0.7797
Val loss:    0.9192
Macro AUROC: 0.5915
Best AUROC:  0.6011
Best loss:   0.9030
Time:        0.9 min

BASELINE FOLD 4 COMPLETE
Best AUROC: 0.6011
Best validation loss: 0.903


In [32]:
# ============================================================
# STEP 4B — BASELINE 5-FOLD OOF
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

BASELINE_ROOT = Path(
    "/kaggle/working/rsna_score_improvement/baseline_cv"
)

BASELINE_OOF_DIR = Path(
    "/kaggle/working/rsna_score_improvement/baseline_oof"
)

BASELINE_OOF_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# Load best-by-AUROC study predictions
# ============================================================

parts = []

for fold in range(5):

    path = (
        BASELINE_ROOT /
        f"fold_{fold}" /
        "best_by_auc_studies.csv"
    )

    assert path.exists(), (
        f"Missing Fold {fold}: {path}"
    )

    df = pd.read_csv(path)

    df["Fold"] = fold

    parts.append(df)

    print(
        f"Fold {fold}: "
        f"{len(df)} studies loaded"
    )


# ============================================================
# Combine OOF
# ============================================================

baseline_oof = pd.concat(
    parts,
    ignore_index=True
)

print("\n" + "=" * 90)
print("BASELINE OOF SAFETY CHECK")
print("=" * 90)

print(
    "Rows:",
    len(baseline_oof)
)

print(
    "Unique studies:",
    baseline_oof[
        "StudyInstanceUID"
    ].nunique()
)

print("\nStudies per fold:")

print(
    baseline_oof["Fold"]
    .value_counts()
    .sort_index()
)

assert len(baseline_oof) == 58

assert (
    baseline_oof[
        "StudyInstanceUID"
    ].nunique()
    == 58
)

assert (
    baseline_oof[
        "StudyInstanceUID"
    ].duplicated()
    .sum()
    == 0
)

print(
    "\n✓ Every study has exactly "
    "one baseline OOF prediction"
)


# ============================================================
# Calculate target AUROCs
# ============================================================

metric_rows = []

for target in LABEL_COLS:

    y_true = baseline_oof[
        f"{target}_true"
    ].values

    y_prob = baseline_oof[
        f"{target}_prob"
    ].values

    auc = roc_auc_score(
        y_true,
        y_prob
    )

    metric_rows.append({
        "Target": target,
        "AUROC": auc
    })


baseline_metrics = pd.DataFrame(
    metric_rows
)

baseline_macro_auc = float(
    baseline_metrics[
        "AUROC"
    ].mean()
)


# ============================================================
# Display
# ============================================================

print("\n" + "=" * 90)
print("BASELINE 58-STUDY OOF RESULTS")
print("=" * 90)

display(
    baseline_metrics.round(4)
)

print(
    "\nBASELINE OOF MACRO AUROC:",
    round(
        baseline_macro_auc,
        4
    )
)


# ============================================================
# Compare directly with V2
# ============================================================

print("\n" + "=" * 90)
print("V2 VS ORIGINAL-STYLE")
print("=" * 90)

comparison = pd.DataFrame({
    "Pipeline": [
        "V2 staged training",
        "Original-style baseline"
    ],
    "OOF_Macro_AUROC": [
        0.5432,
        baseline_macro_auc
    ]
})

comparison["Difference_vs_V2"] = (
    comparison["OOF_Macro_AUROC"]
    - 0.5432
)

display(
    comparison.round(4)
)


# ============================================================
# Save
# ============================================================

baseline_oof.to_csv(
    BASELINE_OOF_DIR /
    "baseline_oof_study_predictions.csv",
    index=False
)

baseline_metrics.to_csv(
    BASELINE_OOF_DIR /
    "baseline_oof_target_metrics.csv",
    index=False
)

pd.DataFrame({
    "Metric": [
        "Macro_AUROC"
    ],
    "Value": [
        baseline_macro_auc
    ]
}).to_csv(
    BASELINE_OOF_DIR /
    "baseline_oof_summary.csv",
    index=False
)

print("\n✓ Baseline OOF files saved")

Fold 0: 12 studies loaded
Fold 1: 12 studies loaded
Fold 2: 12 studies loaded
Fold 3: 11 studies loaded
Fold 4: 11 studies loaded

BASELINE OOF SAFETY CHECK
Rows: 58
Unique studies: 58

Studies per fold:
Fold
0    12
1    12
2    12
3    11
4    11
Name: count, dtype: int64

✓ Every study has exactly one baseline OOF prediction

BASELINE 58-STUDY OOF RESULTS


,Target,AUROC
0,ACL,0.6311
1,MCL,0.6757
2,Medial Meniscus,0.6839
3,Lateral Meniscus,0.6472
4,Medial OA,0.6248
5,Lateral OA,0.6673
6,PF OA,0.5792
7,Effusion,0.7143
8,Synovitis,0.6117
9,Baker's,0.5199



BASELINE OOF MACRO AUROC: 0.6103

V2 VS ORIGINAL-STYLE


,Pipeline,OOF_Macro_AUROC,Difference_vs_V2
0,V2 staged training,0.5432,0.0000
1,Original-style baseline,0.6103,0.0671



✓ Baseline OOF files saved


In [33]:
# ============================================================
# STEP 4C — CHECKPOINT SELECTION COMPARISON
# Best AUROC vs Best Validation Loss
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

BASELINE_ROOT = Path(
    "/kaggle/working/rsna_score_improvement/baseline_cv"
)

# ============================================================
# Helper
# ============================================================

def build_oof_from_files(filename):

    parts = []

    for fold in range(5):

        path = (
            BASELINE_ROOT /
            f"fold_{fold}" /
            filename
        )

        assert path.exists(), f"Missing: {path}"

        df = pd.read_csv(path)
        df["Fold"] = fold

        parts.append(df)

    oof_df = pd.concat(
        parts,
        ignore_index=True
    )

    assert len(oof_df) == 58

    assert (
        oof_df["StudyInstanceUID"]
        .nunique()
        == 58
    )

    return oof_df


def calculate_oof_auc(oof_df):

    rows = []

    for target in LABEL_COLS:

        auc = roc_auc_score(
            oof_df[f"{target}_true"],
            oof_df[f"{target}_prob"]
        )

        rows.append({
            "Target": target,
            "AUROC": auc
        })

    metrics = pd.DataFrame(rows)

    macro = float(
        metrics["AUROC"].mean()
    )

    return macro, metrics


# ============================================================
# Best-by-AUROC models
# ============================================================

oof_auc_selected = build_oof_from_files(
    "best_by_auc_studies.csv"
)

auc_selected_macro, auc_selected_metrics = (
    calculate_oof_auc(
        oof_auc_selected
    )
)


# ============================================================
# Best-by-validation-loss models
# ============================================================

oof_loss_selected = build_oof_from_files(
    "best_by_loss_studies.csv"
)

loss_selected_macro, loss_selected_metrics = (
    calculate_oof_auc(
        oof_loss_selected
    )
)


# ============================================================
# Comparison
# ============================================================

comparison = pd.DataFrame({
    "Checkpoint": [
        "Best validation loss",
        "Best macro AUROC"
    ],

    "OOF_Macro_AUROC": [
        loss_selected_macro,
        auc_selected_macro
    ]
})

comparison["Difference"] = (
    comparison["OOF_Macro_AUROC"]
    - loss_selected_macro
)

print("=" * 85)
print("CHECKPOINT SELECTION COMPARISON")
print("=" * 85)

display(
    comparison.round(4)
)


# ============================================================
# Per-target comparison
# ============================================================

target_compare = (
    loss_selected_metrics
    .rename(
        columns={
            "AUROC":
                "Best_Loss_AUROC"
        }
    )
    .merge(
        auc_selected_metrics.rename(
            columns={
                "AUROC":
                    "Best_AUC_AUROC"
            }
        ),
        on="Target"
    )
)

target_compare["Difference"] = (
    target_compare["Best_AUC_AUROC"]
    - target_compare["Best_Loss_AUROC"]
)

print("\nPer-target:")
display(
    target_compare.round(4)
)

CHECKPOINT SELECTION COMPARISON


,Checkpoint,OOF_Macro_AUROC,Difference
0,Best validation loss,0.6217,0.0000
1,Best macro AUROC,0.6103,-0.0115



Per-target:


,Target,Best_Loss_AUROC,Best_AUC_AUROC,Difference
0,ACL,0.6360,0.6311,-0.0049
1,MCL,0.6961,0.6757,-0.0204
2,Medial Meniscus,0.7139,0.6839,-0.0300
3,Lateral Meniscus,0.6509,0.6472,-0.0037
4,Medial OA,0.6651,0.6248,-0.0403
5,Lateral OA,0.7021,0.6673,-0.0348
6,PF OA,0.5354,0.5792,0.0438
7,Effusion,0.6907,0.7143,0.0236
8,Synovitis,0.6977,0.6117,-0.0860
9,Baker's,0.5036,0.5199,0.0163


In [34]:
# ============================================================
# STEP 4D — BEST-LOSS OOF AGGREGATION TEST
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

BASELINE_ROOT = Path(
    "/kaggle/working/rsna_score_improvement/baseline_cv"
)

LOSS_OOF_DIR = Path(
    "/kaggle/working/rsna_score_improvement/baseline_loss_oof"
)

LOSS_OOF_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# Load best-by-loss triplet predictions from all folds
# ============================================================

parts = []

for fold in range(5):

    path = (
        BASELINE_ROOT /
        f"fold_{fold}" /
        "best_by_loss_triplets.parquet"
    )

    assert path.exists(), f"Missing: {path}"

    df = pd.read_parquet(path)

    df["Fold"] = fold

    parts.append(df)

    print(
        f"Fold {fold}: "
        f"{len(df)} triplets, "
        f"{df['StudyInstanceUID'].nunique()} studies"
    )

loss_triplets = pd.concat(
    parts,
    ignore_index=True
)

assert (
    loss_triplets["StudyInstanceUID"]
    .nunique()
    == 58
)

print("\nTotal studies: 58")


# ============================================================
# Helpers
# ============================================================

def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))


def score_method(df):

    rows = []

    for target in LABEL_COLS:

        auc = roc_auc_score(
            df[f"{target}_true"],
            df[f"{target}_prob"]
        )

        rows.append({
            "Target": target,
            "AUROC": auc
        })

    metrics = pd.DataFrame(rows)

    return (
        metrics["AUROC"].mean(),
        metrics
    )


# ============================================================
# Mean-logit
# ============================================================

def agg_mean_logit(df):

    rows = []

    for uid, g in df.groupby(
        "StudyInstanceUID"
    ):

        row = {
            "StudyInstanceUID": uid
        }

        for target in LABEL_COLS:

            row[f"{target}_true"] = (
                g[f"{target}_true"].iloc[0]
            )

            row[f"{target}_prob"] = sigmoid_np(
                g[f"{target}_logit"].mean()
            )

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# Series-balanced
# ============================================================

def agg_series_balanced(df):

    rows = []

    for uid, g in df.groupby(
        "StudyInstanceUID"
    ):

        row = {
            "StudyInstanceUID": uid
        }

        for target in LABEL_COLS:

            series_logits = (
                g.groupby(
                    "SeriesInstanceUID"
                )[f"{target}_logit"]
                .mean()
            )

            row[f"{target}_true"] = (
                g[f"{target}_true"].iloc[0]
            )

            row[f"{target}_prob"] = sigmoid_np(
                series_logits.mean()
            )

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# Plane-balanced
# triplets -> series -> plane -> study
# ============================================================

def agg_plane_balanced(df):

    rows = []

    for uid, g in df.groupby(
        "StudyInstanceUID"
    ):

        row = {
            "StudyInstanceUID": uid
        }

        for target in LABEL_COLS:

            series_df = (
                g.groupby(
                    [
                        "Anatomical_Plane",
                        "SeriesInstanceUID"
                    ]
                )[f"{target}_logit"]
                .mean()
                .reset_index()
            )

            plane_logits = (
                series_df
                .groupby(
                    "Anatomical_Plane"
                )[f"{target}_logit"]
                .mean()
            )

            row[f"{target}_true"] = (
                g[f"{target}_true"].iloc[0]
            )

            row[f"{target}_prob"] = sigmoid_np(
                plane_logits.mean()
            )

        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# Compare
# ============================================================

methods = {
    "Mean logit":
        agg_mean_logit(loss_triplets),

    "Series balanced":
        agg_series_balanced(loss_triplets),

    "Plane balanced":
        agg_plane_balanced(loss_triplets),
}

summary = []
target_rows = []

for name, predictions in methods.items():

    macro, metrics = score_method(
        predictions
    )

    summary.append({
        "Method": name,
        "Macro_AUROC": macro
    })

    for row in metrics.itertuples(
        index=False
    ):

        target_rows.append({
            "Method": name,
            "Target": row.Target,
            "AUROC": row.AUROC
        })


summary_df = (
    pd.DataFrame(summary)
    .sort_values(
        "Macro_AUROC",
        ascending=False
    )
    .reset_index(drop=True)
)

target_df = pd.DataFrame(
    target_rows
)

print("\n" + "=" * 85)
print("BEST-LOSS OOF AGGREGATION")
print("=" * 85)

display(
    summary_df.round(4)
)

print("\nPer-target:")

display(
    target_df
    .pivot(
        index="Target",
        columns="Method",
        values="AUROC"
    )
    .round(3)
)

summary_df.to_csv(
    LOSS_OOF_DIR /
    "aggregation_summary.csv",
    index=False
)

print("\n✓ Saved")

Fold 0: 2194 triplets, 12 studies
Fold 1: 2489 triplets, 12 studies
Fold 2: 1627 triplets, 12 studies
Fold 3: 1541 triplets, 11 studies
Fold 4: 2005 triplets, 11 studies

Total studies: 58

BEST-LOSS OOF AGGREGATION


,Method,Macro_AUROC
0,Mean logit,0.6217
1,Series balanced,0.6138
2,Plane balanced,0.6129



Per-target:


Method,Mean logit,Plane balanced,Series balanced
Target,,,
ACL,0.636,0.619,0.621
Baker's,0.504,0.491,0.504
Contusion,0.484,0.507,0.474
Effusion,0.691,0.758,0.727
Fracture,0.485,0.501,0.457
Lateral Meniscus,0.651,0.624,0.639
Lateral OA,0.702,0.665,0.681
MCL,0.696,0.687,0.714
Medial Meniscus,0.714,0.665,0.701



✓ Saved


In [35]:
# ============================================================
# STEP 5A — TEST DATA PREFLIGHT
# ============================================================

from pathlib import Path
import pandas as pd

SUP_DIR = Path(
    "/kaggle/input/notebooks/elliotyang37/"
    "rsna-knee-mri-supervised-fine-tuning/supervised"
)

TEST_MANIFEST_PATH = (
    SUP_DIR / "test_manifest.parquet"
)

TEST_PERCENTILES_PATH = (
    SUP_DIR / "test_series_percentiles.csv"
)

print("=" * 85)
print("TEST INFERENCE PREFLIGHT")
print("=" * 85)

print(
    "Test manifest exists:",
    TEST_MANIFEST_PATH.exists()
)

print(
    "Test percentiles exist:",
    TEST_PERCENTILES_PATH.exists()
)

test_manifest = pd.read_parquet(
    TEST_MANIFEST_PATH
)

test_percentiles = pd.read_csv(
    TEST_PERCENTILES_PATH
)

print("\nTest manifest shape:")
print(test_manifest.shape)

print("\nTest manifest columns:")
print(test_manifest.columns.tolist())

print("\nUnique studies:")
print(
    test_manifest[
        "StudyInstanceUID"
    ].nunique()
)

print("\nUnique series:")
print(
    test_manifest[
        "SeriesInstanceUID"
    ].nunique()
)

print("\nPlanes:")
print(
    test_manifest[
        "Anatomical_Plane"
    ].value_counts()
)

print("\nPercentile shape:")
print(test_percentiles.shape)

print("\nPercentile columns:")
print(test_percentiles.columns.tolist())


# ============================================================
# Check all 5 final checkpoints
# ============================================================

BASELINE_ROOT = Path(
    "/kaggle/working/"
    "rsna_score_improvement/baseline_cv"
)

print("\n" + "=" * 85)
print("FINAL 5 MODEL CHECK")
print("=" * 85)

for fold in range(5):

    path = (
        BASELINE_ROOT /
        f"fold_{fold}" /
        "best_by_loss_model.pt"
    )

    print(
        f"Fold {fold}: "
        f"{'✓' if path.exists() else 'MISSING'}"
    )

TEST INFERENCE PREFLIGHT
Test manifest exists: True
Test percentiles exist: True

Test manifest shape:
(527, 8)

Test manifest columns:
['StudyInstanceUID', 'SeriesInstanceUID', 'Anatomical_Plane', 'PreviousPath', 'CentrePath', 'NextPath', 'P01', 'P99']

Unique studies:
3

Unique series:
15

Planes:
Anatomical_Plane
Axial       259
Sagittal    166
Coronal     102
Name: count, dtype: int64

Percentile shape:
(15, 6)

Percentile columns:
['StudyInstanceUID', 'SeriesInstanceUID', 'Anatomical_Plane', 'P01', 'P99', 'ValidSlices']

FINAL 5 MODEL CHECK
Fold 0: ✓
Fold 1: ✓
Fold 2: ✓
Fold 3: ✓
Fold 4: ✓
